# Assignment 2 - Baselines and Leakage-Safe Features

This notebook defines the feature and baseline layer for hourly bus-level `pd` forecasting. It uses time-ordered windows only and treats `forecast_created_at` as the information boundary.

## Leakage Rules

- No random train/test splitting.
- No centered rolling windows.
- Rolling statistics are computed from `pd.shift(1)` before rolling.
- Validation historical averages are mapped from the training window only.
- Forecast rows mask lag features whose source timestamp is after `forecast_created_at`.
- 2025 remains held out for the final test workflow.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "data").exists() and (candidate / "solution" / "assignment_2" / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not find ECESIS repository root from notebook path.")

src_dir = repo_root / "solution" / "assignment_2" / "src"
outputs_dir = repo_root / "solution" / "assignment_2" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(src_dir))

pd.set_option("display.max_columns", None)


In [ ]:
from evaluation import make_fold_table
from load_data import first_buses_from_first_batch, read_bus
from features import build_bus_feature_frame, apply_forecast_boundary, fit_group_mean
from baselines import bus_baseline_predictions, long_baseline_frame

folds = make_fold_table()
folds.to_csv(outputs_dir / "walk_forward_results.csv", index=False)
folds

## Prototype Feature Window

The prototype fold uses a deterministic subset of high-volume buses from the first 2022 batch. This keeps notebook execution practical while preserving hourly bus-level granularity.

In [ ]:
bus_ids = first_buses_from_first_batch(2022, n_buses=20)
bus = read_bus(
    [2022],
    columns=["bus_unique_id", "bus_type", "base_kv", "zone_name", "pd", "pg", "date", "he"],
    start_date="2022-01-01",
    end_date="2022-04-30",
    bus_ids=bus_ids,
)

bus_features = build_bus_feature_frame(bus)
feature_audit = pd.DataFrame({
    "rows": [len(bus_features)],
    "buses": [bus_features["bus_unique_id"].nunique()],
    "zones": [bus_features["zone_name"].nunique()],
    "min_timestamp": [bus_features["timestamp"].min()],
    "max_timestamp": [bus_features["timestamp"].max()],
})
feature_audit.to_csv(outputs_dir / "feature_audit_prototype.csv", index=False)
feature_audit

## Baselines

The baseline outputs include previous-week same-hour and training-window historical mean by bus, HE, and day-of-week.

In [ ]:
train = bus_features[bus_features["date"] <= pd.Timestamp("2022-03-31")].copy()
validation = bus_features[(bus_features["date"] >= pd.Timestamp("2022-04-01")) & (bus_features["date"] <= pd.Timestamp("2022-04-30"))].copy()

history = fit_group_mean(train, ["bus_unique_id", "he", "day_of_week"], "pd", "historical_avg_bus_he_dow_pd")
validation = validation.drop(columns=["historical_avg_bus_he_dow_pd"], errors="ignore").merge(
    history, on=["bus_unique_id", "he", "day_of_week"], how="left"
)
validation = apply_forecast_boundary(validation, "2022-03-31 23:00:00")

baseline_wide = bus_baseline_predictions(validation)
baseline_out = long_baseline_frame(baseline_wide)

frames = []
for horizon, frame in [("next_day", baseline_out[baseline_out["date"] == pd.Timestamp("2022-04-01")]), ("next_month", baseline_out)]:
    tmp = frame.copy()
    tmp["horizon"] = horizon
    frames.append(tmp)
baseline_out = pd.concat(frames, ignore_index=True)
baseline_out.to_csv(outputs_dir / "baseline_forecasts.csv", index=False)
baseline_out.head()